In [1]:
%run 00_setup.ipynb

In [2]:
RUN_ID = REFERENCE_RUN

In [3]:
print(RUN_ID)

2


In [4]:
ann    = load("annotations.csv", run=RUN_ID)
gold   = load("gold.csv", run=RUN_ID)
status = load("status.csv", run=RUN_ID)
fail   = load("failures.csv", run=RUN_ID)
dist   = load("pattern_distribution.csv", run=RUN_ID)
sample = load("sample.csv", run=RUN_ID)
meta   = load("meta.csv", run=RUN_ID).iloc[0].to_dict()

print(f"run {RUN_ID}: {meta['label']}  |  painel: {meta['panel']}")
print(f"{len(ann)} linhas de anotacao, {len(gold)} linhas de gold, {len(status)} linhas de estado, {len(fail)} linhas de falha")

run 2: llm_panel #2  |  painel: deepseek/deepseek-v4-flash;meta-llama/llama-3.3-70b-instruct;openai/gpt-oss-120b;qwen/qwen3-next-80b-a3b-instruct
67317 linhas de anotacao, 5000 linhas de gold, 3600 linhas de estado, 57 linhas de falha


## Taxa de falha de *parsing* por modelo
O painel falha *por modelo*: cada modelo pode devolver uma saída que não se consegue interpretar. É a primeira leitura de fiabilidade.

In [5]:
status_by_model(status)

,parse_err,total,pct
modelSlug,,,
meta-llama/llama-3.3-70b-instruct,47,900,5.22
openai/gpt-oss-120b,8,900,0.89
deepseek/deepseek-v4-flash,1,900,0.11
qwen/qwen3-next-80b-a3b-instruct,1,900,0.11


### Razões de paragem das falhas

Um `finish_reason = stop` significa que o modelo terminou por iniciativa própria

In [6]:
status.loc[status["status"] != "completed", "finishReason"].value_counts()

finishReason
stop     55
error     2
Name: count, dtype: int64

### Painéis parciais

In [7]:
print("partial-panel reviews:        %d / %d (%.1f%%)" % partial_panel(status))
print("adjudicated (open) on partial: %d / %d (%.1f%%)" % adjudicated_on_partial(status, gold, "open"))

partial-panel reviews:        57 / 900 (6.3%)
adjudicated (open) on partial: 16 / 231 (6.9%)


## E se excluirmos as revisões de painel parcial?
As avaliações adjudicadas em painel parcial ficam na análise (a maioria estrita é relativa aos membros que votaram). Ainda assim, importa medir o efeito recalculando-se o F1 macro sem essas avaliações e comparando-se com o valor antes da exclusão. Se os valores mal se moverem, os painéis parciais são inócuos.

In [8]:
bad = set(status.loc[status["status"] != "completed", "individualId"])
cal_full = cells(ann, gold, "open")
cal_sem  = cal_full[~cal_full["individualId"].isin(bad)]
comp = pd.DataFrame({"com revisões parciais": macro(calibration(cal_full))["f1"],
                     "sem revisões parciais": macro(calibration(cal_sem))["f1"]}).round(3)
print(comp.to_string())

                                   com revisões parciais  sem revisões parciais
model                                                                          
deepseek/deepseek-v4-flash                         0.204                  0.194
meta-llama/llama-3.3-70b-instruct                  0.272                  0.270
openai/gpt-oss-120b                                0.179                  0.175
panel-majority                                     0.194                  0.181
qwen/qwen3-next-80b-a3b-instruct                   0.446                  0.442
